#### **Structured Output**
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


#### **Pydantic**

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [9]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model(model="groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.8', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000029A8D8D8DD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029A8D8DA600>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [10]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.8', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000029A8D8D8DD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029A8D8DA600>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [6]:
model.invoke("Provide details about the movie: Forrest Gump")

AIMessage(content='**Forrest\u202fGump – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Forrest Gump* |\n| **Release Year** | 1994 |\n| **Genre** | Drama / Romance / Comedy (often classified as a “dramedy”) |\n| **Running Time** | 142 minutes |\n| **Director** | Robert Zemeckis |\n| **Screenwriter** | Eric Roth (adapted from the 1986 novel by Winston\u202fGraham) |\n| **Producer(s)** | Wendy Finerman, Steve Tisch, Steve Starkey, Vinton\u202fKehl, and Robert Zemeckis |\n| **Studio** | Paramount Pictures (distribution) |\n| **Music** | Alan\u202fSilvestri (original score) – notable soundtrack includes classic 1950‑70s hits (e.g., “Fortunate Son,” “Mrs. Robinson,” “Sweet Home Alabama”) |\n| **Cinematography** | Don\u202fBacon |\n| **Editor** | Arthur\u202fSchrader |\n| **Budget** | Approx. **$55\u202fmillion** |\n| **Box‑Office Gross** | About **$677\u202fmillion** worldwide (making it the highest‑grossing film of 1994) |\n| **MPAA Rating** | PG‑13 (for 

In [11]:
response=model_with_structure.invoke("Provide details about the movie: Forrest Gump")
response

Movie(title='Forrest Gump', year=1994, director='Robert Zemeckis', rating=8.8)

#### **Message output with parsed structure**

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

###include_raw=True tells LangChain to return both the original model response and the parsed structured result.
model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response=model_with_structure.invoke("Provide details about the movie: Forrest Gump")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Forrest Gump. We can use the function Movie to provide details: director, rating, title, year. Need to fill in appropriate data. Forrest Gump directed by Robert Zemeckis, rating maybe 8.8/10 (IMDb) or 8.5. Use rating out of 10. Year 1994. Title "Forrest Gump". So call function.', 'tool_calls': [{'id': 'fc_2dbecb5f-d688-4b4d-9147-8f5ee2ab1d7b', 'function': {'arguments': '{"director":"Robert Zemeckis","rating":8.8,"title":"Forrest Gump","year":1994}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 166, 'total_tokens': 310, 'completion_time': 0.300857016, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.00628553, 'prompt_tokens_details': None, 'queue_time': 0.229915142, 'total_time': 0.307142546}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on

#### **Nested Structure**

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    """An actor with details."""
    name: str = Field(..., description="The name of the actor")
    role:str = Field(..., description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    """A movie with details and actors."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    cast: str = Field(..., description="The cast of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")
    actors: list[Actor] = Field(..., description="A list of actors in the movie")
    budget: float = Field(..., description="The budget of the movie in millions of USD")


model_with_structure = model.with_structured_output(MovieDetails) 

reponse = model_with_structure.invoke("Provide details about the movie: Forrest Gump")
reponse

MovieDetails(title='Forrest Gump', year=1994, cast='Tom Hanks, Robin Wright, Gary Sinise, Sally Field, Mykelti Williamson, Haley Joel Osment (young Forrest), Michael Conner Humphreys (young Forrest), Rebecca Williams (young Jenny)', rating=8.8, actors=[Actor(name='Tom Hanks', role='Forrest Gump'), Actor(name='Robin Wright', role='Jenny Curran'), Actor(name='Gary Sinise', role='Lt. Dan Taylor'), Actor(name='Sally Field', role='Mrs. Gump'), Actor(name='Mykelti Williamson', role='Bubba'), Actor(name='Haley Joel Osment', role='Young Forrest Gump'), Actor(name='Michael Conner Humphreys', role='Young Forrest Gump (earlier scenes)'), Actor(name='Rebecca Williams', role='Young Jenny Curran')], budget=55.0)

#### **TypeDict**

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [22]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie: Fight Club")
response

{'director': 'David Fincher',
 'rating': 8.8,
 'title': 'Fight Club',
 'year': 1999}

In [23]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie: Fight Club")
response

{'budget': 63000000,
 'cast': [{'name': 'Brad Pitt', 'role': 'Tyler Durden'},
  {'name': 'Edward Norton', 'role': 'The Narrator'},
  {'name': 'Helena Bonham Carter', 'role': 'Marla Singer'},
  {'name': 'Jared Leto', 'role': 'Angel Face'},
  {'name': 'Meat Loaf', 'role': "Robert 'Bob' Paulson"},
  {'name': 'Zach Grenier', 'role': 'Richard Chesler'},
  {'name': 'Richmond Arquette', 'role': 'Thomas'},
  {'name': 'Eion Bailey', 'role': 'Walter'},
  {'name': 'David Andrews', 'role': 'Chief of Police'}],
 'genres': ['Drama', 'Thriller'],
 'title': 'Fight Club',
 'year': 1999}

In [24]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

#### **DataClass**
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [2]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [4]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='48170642-6122-4f60-b83b-9f5e0e705fcd'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user asks to extract contact info from a string. The system says we need to output ONLY the final JSON object, adhering to ContactInfo schema: name, email, phone all required. So we must produce JSON like {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}.\n\nWe must ensure compact JSON formatting (no spaces?). "compact JSON formatting" means no unnecessary whitespace. So produce: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}.\n\nWe need to verify JSON is valid. Yes.\n\nThus final answer.'}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 238, 'total_tokens': 404, 'completion_time

In [5]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"] 
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [8]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')